In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import os

def prepare():
    module_path = os.path.abspath(os.path.join('../','../'))
    if module_path not in sys.path:
        sys.path.append(module_path)

In [ ]:
import torch
import numpy as np
prepare()
from exp_labelcert_binaryclass import run

In [ ]:
seeds = [0, 1, 2, 3, 4]
deltas = [0.00, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]

In [ ]:
model_params = dict(
    label = "GCN", 
    model = "GCN", 
    normalization = "row_normalization",
    activation = "relu",
    depth = 1,
    regularizer = 0.01,
    pred_method = "svm",
    bias = False,
    alpha_tol = 1e-4,
    solver = "qplayer",
)

certificate_params = dict(
    delta = 0.01,
    TimeLimit = 86400,
    LogToConsole = 1,
    OutputFlag = 1,
    Threads = 2,
    Presolve = 2
)

verbosity_params = dict(
    debug_lvl = "warning"
)  

other_params = dict(
    device = "0",
    dtype = torch.float64,
    allow_tf32 = False,
    path_gurobi_license="/mnt/c/Users/emiel/gurobi.lic"
)

In [ ]:
data_params = dict(
    dataset = "csbm",
    learning_setting = "transductive", 
    specification = dict(
        classes = 2,
        n_trn_labeled = 10,
        n_trn_unlabeled = 0,
        n_val = 10,
        n_test = 180,
        sigma = 1,
        avg_within_class_degree = 1.58 * 2,
        avg_between_class_degree = 0.37 * 2,
        K = 1.5,
        # seed = 0 # used to generate the dataset & data split
    )
)

In [ ]:
import pandas as pd
import time

metrics_to_log = ["accuracy_test", "accuracy_trn", "accuracy_cert_pois_robust", "delta", "runtime"]
output_base_dir = "results/collective"
os.makedirs(output_base_dir, exist_ok=True)

dataset_name = "csbm"

for delta in deltas:
    certificate_params["delta"] = delta
    summary_results = []
    print(f"  Running for delta: {delta:.2f}")

    for seed in seeds:
        data_params["specification"]["seed"] = seed
        start_time = time.time()
        result = run(data_params=data_params, model_params=model_params,
                     certificate_params=certificate_params, verbosity_params=verbosity_params,
                     other_params=other_params, seed=seed)
        runtime = round(time.time() - start_time, 2)
        result['runtime'] = runtime
        summary_results.append({k: result.get(k) for k in metrics_to_log})

    df = pd.DataFrame(summary_results, index=[f"Seed {s}" for s in seeds])
    df.index.name = "seed"
    output_filename = f"{dataset_name}-{delta:.2f}.csv"
    output_path = os.path.join(output_base_dir, output_filename)
    df.to_csv(output_path, index=True)

    print(f"  Results saved to: {output_path}")
    print(df)
    print("-" * 30)
